In [47]:
from qat.fermion.hamiltonians import FermionHamiltonian, SpinHamiltonian
from qat.fermion.hamiltonians import make_anderson_model
from qat.fermion.trotterisation import make_trotterisation_routine
from qat.lang import *
from qat.core import Term, Schedule, Variable, Observable
from qat.qpus import get_default_qpu
from qat.plugins import ScipyMinimizePlugin
from scipy.optimize import minimize
from qat.lang.AQASM import Program, QRoutine

import numpy as np
import numpy.typing as npt

import matplotlib.pyplot as plt
from collections import deque

from scipy.optimize import minimize

In [ ]:
prog = Program()
qubit = prog.qalloc(4)
routine = QRoutine()
H_test= SpinHamiltonian(4, [Term(1, "XXXX", [0,1,2,3])])

trot = make_trotterisation_routine(H_test, n_trotter_steps=2, final_time=4.5)
prog.apply(trot, qubit[0], qubit[1], qubit[2], qubit[3])

circuit = prog.to_circ()
qpu = get_default_qpu()
job3 = circuit.to_job(observable = Observable(4, pauli_terms = [Term(1, "Z", [0])]))
result = qpu.submit(job3)

print(result.value)



-0.9111302618846742


In [ ]:
'''
routine.apply(routX, [0,1])
a = make_trotterisation_routine((-1)*AndersonHamiltonian, n_trotter_steps=2, final_time=1)
routine.apply(a,[range(4)])
routine.apply(routX_dag, 0)
control = routine.ctrl()

prog.apply(H, 0)
prog.apply(S.dag(), 0)
prog.apply(X, 1)
prog.apply(control, [0,1,2,3,4])
prog.apply(H, 0)
'''

In [57]:
from qat.lang import Program, H, CNOT

# Create a Program
qprog = Program()
# Number of qbits
nbqbits = 2
# Allocate some qbits
qbits = qprog.qalloc(nbqbits)

# Apply some quantum Gates
H(qbits[0])
CNOT(qbits[0], qbits[1])

# Export this program into a quantum circuit
circuit = qprog.to_circ()

# Import a Quantum Processor Unit Factory (the default one)
from qat.qpus import get_default_qpu

# Create a Quantum Processor Unit
qpu = get_default_qpu()

obs = Observable(2, pauli_terms= [Term(1, "ZZ", [0,1])])
# Create a job
job = circuit.to_job(observable = obs)


# Submit the job to the QPU
result = qpu.submit(job)

# Iterate over the final state vector to get all final components
print(result.value)

0.9999999999999998


In [58]:
from qat.lang.AQASM import Program, H, CNOT
from qat.qpus import get_default_qpu

# Create a circuit
qprog = Program()
qbits = qprog.qalloc(2)
H(qbits[0])
CNOT(qbits[0], qbits[1])
circuit = qprog.to_circ()

# Create an observable
from qat.core import Observable, Term
obs = Observable(2, pauli_terms=[Term(1, "XZ", [0, 1])])

# Create a job
job = circuit.to_job(observable=obs)

# Execute
result = get_default_qpu().submit(job)
print("<O> = ", result.value)

<O> =  0.0


On commence d'abord par définir l'opérateur $c^{\dagger}_{i, \sigma1}c_{j, \sigma2}$ qui va être utilisé pour définir le hamiltonien :

In [2]:
#Definition de l'operateur  
def couplingOp(n_qubits: int, energy: float, i: int, j: int, spin_i: str, spin_j: str) : 
    """ Calcule une instance de FermionHamiltonian associée à l'opérateur c^{dagger}_{i, \sigma1}c_{j, \sigma2}, 
    i et j sont les numéros de site, energy est le préfacteur énergétique apparaissant dans 
    la définition du hamiltonien. """
    if spin_i == "+" :
         if spin_j == "-" :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i, j+1])])
         else :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i, j])])
    else :
         if spin_j == "-" :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i+1, j+1])])
         else :
            return FermionHamiltonian(n_qubits, [Term(energy, "Cc", [i+1, j])]) 
        
def nOp(n_qubits, energy, i, spin) :
    """Calcule une instance de FermionHamiltonian associée à l'opérateur n_{i, spin} avec le préfacteur energy."""
    return couplingOp(n_qubits, energy, i, i, spin, spin)
    

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
C:\Users\elkha\AppData\Local\Temp\ipykernel_17924\3843252086.py:3: SyntaxWarning: invalid escape sequence '\s'
  """ Calcule une instance de FermionHamiltonian associée à l'opérateur c^{dagger}_{i, \sigma1}c_{j, \sigma2},


In [3]:
print(couplingOp(4,2,1,2,"+","-"))
print(type(couplingOp(4,2,1,2,"+","-")))


2 * (Cc|[1, 3])
<class 'qat.fermion.hamiltonians.FermionHamiltonian'>


Definition de l'Hamiltonien de Anderson (avec la possibilité de choisir le nombre d'impuretés contrairement à la fonction make_anderson_model de myQLM)

In [4]:
#Definition de l'hamiltonien
def hamiltonian(n_impurities: int, n_bain: int, U: float, mu: float, t: float, V, epsilon: list) :
    """ Calcule le hamiltonien du modèle d'Anderson et le renvoie sous la forme 
    du couple (SpinHamiltonian, matrice associée)."""
    n_qubits = 2 * n_bain + 2 * n_impurities
    H_repulsion = 0
    H_hopping = 0
    H_chem = 0
    for wire in range(0, 2*n_impurities, 2) :
        #Calcul du terme de potentiel chimique
        H_chem += nOp(n_qubits, -mu, wire, "+")
        H_chem += nOp(n_qubits, -mu, wire, "-")

        #Calcul du terme d'effet tunnel entre impuretés, dit "hopping"
        for wire2 in range(0, 2*n_impurities, 2) :
            if wire2 != wire :
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "+", "-")
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "+", "+")
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "-", "-")
                H_hopping += couplingOp(n_qubits, -t, wire, wire2, "-", "+")
            else :
                #Calcul du terme de répulsion coulombienne
                H_repulsion += nOp(n_qubits, U, wire, "+")*nOp(n_qubits, 1, wire,"-")
    
    H_impurity_bath = 0
    H_bath = 0
    for wire3 in range(2*n_impurities, n_qubits, 2) :
        i = (wire3 - 2*n_impurities) // 2
        e = epsilon[i]
        #Calcul du terme lié au bain
        H_bath += nOp(n_qubits, e, wire3, "+")
        H_bath += nOp(n_qubits, e, wire3, "-")
        
        for wire4 in range(0, 2*n_impurities, 2) :
            #Calcul du terme d'interaction impureté-bain
            j = wire4 // 2
            v = V[j][i]
            
            vstar = v if np.isreal(v) else np.conj(v)
            
            H_impurity_bath += couplingOp(n_qubits, v, wire4, wire3, "+", "+") 
            H_impurity_bath += couplingOp(n_qubits, vstar, wire3, wire4, "+", "+")
            
            H_impurity_bath += couplingOp(n_qubits, v, wire4, wire3, "-", "-") 
            H_impurity_bath += couplingOp(n_qubits, vstar, wire3, wire4, "-", "-")
        

    H_anderson = H_repulsion + H_chem + H_impurity_bath + H_bath + H_hopping
    H_spin = H_anderson.to_spin(method = "jordan-wigner")#Fermion to Qubit
    return H_spin, H_spin.get_matrix()



In [5]:
#Test de correspondance avec la fonction de MyQLM pour une impureté
AndersonHamiltonian, matrix = hamiltonian(1, 1, 2, 4, 3, [[2]], [4]) 
AndersonHamiltonian1, matrix1 = hamiltonian(1, 3, 3, 2, 1, [[2, 1, 5]], [1,0, 4])
qlmA = make_anderson_model(2, 4, [2], [4]).to_spin()
qlmB = make_anderson_model(3, 2, [2, 1, 5], [1, 0, 4]).to_spin()
print(AndersonHamiltonian == qlmA)
print(qlmB == AndersonHamiltonian1)



True
True


Implementation du HVA pour passer d'un hamiltonien connu (diagonalisable) à notre hamiltonien d'Anderson

In [39]:
@qrout
def ansatz(theta):
    " Dummy Ansatz"
    for qbit in range(3):
        RY(theta)(qbit)
        RZ(4 * theta)(qbit)

print(type(ansatz))

<class 'qat.lang.decorator._BaseWrapper'>


In [51]:
rou = QRoutine()
rou.apply(X, 0)
H = SpinHamiltonian(4, [Term(1, "X", [0])])

def hva_ansatz(psi0, depth, hamiltonians,
                h_target, n_steps,  params) : #prend des hamiltoniens de spin et simule un recuit quantique
    rout = QRoutine()
    k = len(hamiltonians)
    rout.apply(psi0)
    for i in range(depth) :
        for j in range(k) :
            rout.apply(make_trotterisation_routine(hamiltonians[j], n_trotter_steps=n_steps,
                                                    final_time = params[depth*i + j]))
            rout.apply(make_trotterisation_routine(h_target, n_trotter_steps=n_steps, 
                                                    final_time = params[depth *i + k]))
    return rout

def cost_function(psi0, depth, hamiltonians,
                h_target, n_steps,  params) :
    
    job = hva_ansatz(psi0, depth, hamiltonians,
                h_target, n_steps,  params).to_job(observable = h_target)
    qpu = get_default_qpu()
    result = qpu.submit(job)
    return result.value

initial_params = np.random.random(2)
variational_circuit = lambda params: cost_function(rou, 1, [[H]], AndersonHamiltonian, 1, params)

ground_state_energy = minimize(variational_circuit, x0 = initial_params)
print(ground_state_energy)



InvalidGateArguments: Gate None of arity 1 cannot be applied on []

In [38]:
variational_circuit = lambda params: hva_ansatz(rou, 1, [[H]], AndersonHamiltonian, 1, params)
print(type(variational_circuit))
job = variational_circuit.to_job(observable=AndersonHamiltonian)
result = qpu.submit(job)
print('final energy:', result.value)
print('best parameters:', result.parameter_map)
print('trace:', result.meta_data['optimization_trace'])


<class 'function'>


AttributeError: 'function' object has no attribute 'to_job'

We now compute the green function for $c_{j,\sigma}(t)$ and $c_{i, \sigma'}^{\dagger}(0)$ : To do that we calculate $\exp{(iHt)} c_{j,\sigma} \exp{(-iHt)} c_{i, \sigma'}^{\dagger} \ket{\psi_{0}}$, and we calculate the overlap of this state with the ground state

Avant toute chose, il faut implémenter une façon de calculer une exponentielle complexe d'une chaîne de Pauli (Pauli string), pour cela on met en place des transformations qui permettent à se ramener à des mots de Pauli uniquement avec des matrices Zi, dont l'exponentielle complexe peut s'implémenter grâce à des CNOT et des rotations d'axe Z, on a en particulier :
$$R_{Yq}\left(\frac{\pi}{2}\right) Z_q R_{Yq}^\dagger\left(\frac{\pi}{2}\right) = X_q$$
$$
R_{Yq}\left(\frac{\pi}{2}\right) e^{-i\theta Z_0 \dots Z_q \dots Z_N} R_{Yq}^\dagger\left(\frac{\pi}{2}\right) =
e^{-i\theta Z_0 \dots X_q \dots Z_N}
$$
$$
R_{Xq}\left(-\frac{\pi}{2}\right) Z_q R_{Xq}^\dagger\left(-\frac{\pi}{2}\right) = Y_q
$$
$$
R_{Xq}\left(-\frac{\pi}{2}\right) e^{-i\theta Z_0 \dots Z_q \dots Z_N} R_{Yq}^\dagger\left(-\frac{\pi}{2}\right) =
e^{-i\theta Z_0 \dots Y_q \dots Z_N}
$$




In [8]:
def expPauli(n_qubits, term, t) :
    """Convertit des exponentielles de produits de matrices de Pauli exp(iX ... Z ... Y) sous la forme 
    exp(i R1 Z1...ZN RN) grâce à des matrices de roatation pour pouvoir compiler l'opérateur term 
    en circuit quantique. """
    before = deque() #pile des rotations qui s'executeront avant le calcul de l'exponentielle de matrice
    after = deque() #file des rotations qui s'executeront après le calcul de l'exponentielle de matrice
    CNOTqueue= deque() #file des CNOT qui relieront les matrices Zi en amont

    prog = Program() #initialize of program 
    qubits = prog.qalloc(n_qubits) #give a qubit register
    
    pauliString, nq = term.op, term.qbits #nq[i] is the qubit upon which pauliString[i] acts
    n = len(pauliString)
    for i in range(n-1) :
        CNOTqueue.append((CNOT, qubits[nq[i]], qubits[nq[n - 1]]))
        if pauliString[i] == "X" :
            before.append((RY(np.pi / 2),qubits[nq[i]]))
            after.append((RY(- np.pi / 2),qubits[nq[i]]))
        elif pauliString[i] == "Y" :
            before.append((RX(-np.pi / 2),qubits[nq[i]]))
            after.append((RX(np.pi / 2),qubits[nq[i]]))
    
    if pauliString[n - 1] == "X" :
        before.append((RY(np.pi / 2), qubits[nq[n - 1]]))
        after.append((RY(- np.pi / 2), qubits[nq[n - 1]]))
    elif pauliString[n - 1] == "Y" :
        before.append((RX(-np.pi / 2), qubits[nq[n - 1]]))
        after.append((RX(np.pi / 2), qubits[nq[n - 1]]))

    
    CNOTstack = CNOTqueue.copy() #pile des CNOT qui relieront les matrices Zi en aval
    print(len(CNOTstack))

    while len(before) > 0 :
        prog.apply(*before.pop())
    while len(CNOTqueue) > 0 :
        prog.apply(*CNOTqueue.popleft())

    prog.apply(RZ(2*t), qubits[nq[n - 1]])

    while len(CNOTstack) > 0 :
        prog.apply(*CNOTstack.pop())
    while len(after) > 0 :
        prog.apply(*after.popleft())
    
    circuit = prog.to_circ()
    return circuit


expPauli(4, AndersonHamiltonian.terms[0], 3).display() #ZZ | 0,1

expPauli(4, AndersonHamiltonian.terms[3], 3).display() #XZX | 0,1,2 

1


2


In [9]:
circuitTest = expPauli(4, AndersonHamiltonian.terms[3], 3)

''' #executer cette partie ci-dessous fait crasher systématiquement le kernel, même le code de la documentation ne marche pas avec 
la méthode get_default_qpu(), submit() etc#

qpu = get_default_qpu
job = circuitTest.to_job(psi_0 = psi0)
result = qpu.submit(job)

finalState = np.array([
(sample.state, sample.amplitude) 
for sample in result
])

print(finalState)
    '''

2


' #executer cette partie ci-dessous fait crasher systématiquement le kernel, même le code de la documentation ne marche pas avec \nla méthode get_default_qpu(), submit() etc#\n\nqpu = get_default_qpu\njob = circuitTest.to_job(psi_0 = psi0)\nresult = qpu.submit(job)\n\nfinalState = np.array([\n(sample.state, sample.amplitude) \nfor sample in result\n])\n\nprint(finalState)\n    '

We also need to take into account the annihilation and creation operators, since they aren't unitaries, we decompose them as sum of unitaries through Jordan-Wigner mapping, we have : 
$$
c_{i,\uparrow} = \frac{1}{2} \left(X_{2i} + i Y_{2i} \right) \prod_{j<2i} Z_j
$$
$$
c_{i,\downarrow} = \frac{1}{2} \left(X_{2i+1} + i Y_{2i+1} \right) \prod_{j<2i+1} Z_j
$$
$$
c_{i,\uparrow}^{\dagger} = \frac{1}{2} \left(X_{2i} - i Y_{2i} \right) \prod_{j<2i} Z_j
$$
$$
c_{i,\downarrow}^{\dagger} = \frac{1}{2} \left(X_{2i+1} - i Y_{2i+1} \right) \prod_{j<2i+1} Z_j
$$

We also have $iY = RY(-\pi) = \begin{pmatrix}
0 & 1 \\
-1 & 0
\end{pmatrix}
$ and $-iY = RY(\pi)$

In [10]:
def cMapping(n_qubits, n_impurity, spin) :
    """Transforme l'opérateur fermionique c_spin en suité d'opérateurs de spins."""
    progX = Program()
    qubitsX = progX.qalloc(n_qubits)
    progY = Program()
    qubitsY = progY.qalloc(n_qubits)
    
    if spin == "+" :
        progX.apply(X, qubitsX[2*n_impurity])
        progY.apply(RY(-np.pi), qubitsY[2*n_impurity]) #+iY
        for wire in range(2 * n_impurity) : 
            progX.apply(Z, qubitsX[wire])
            progY.apply(Z, qubitsY[wire])
    else : #spin down
        progX.apply(X, qubitsX[2*n_impurity + 1])
        progY.apply(RY(-np.pi), qubitsY[2*n_impurity + 1])
        for wire in range(2 * n_impurity) :
            progX.apply(Z, qubitsX[wire])
            progY.apply(Z, qubitsY[wire])

    circuitX = progX.to_circ()
    circuitY = progY.to_circ()
    return circuitX, circuitY
    

In [11]:
def cDagMapping(n_qubits, n_impurity, spin) :
    """Transforme l'opérateur fermionique c^dagger_spin en suité d'opérateurs de spins."""
    progX = Program()
    qubitsX = progX.qalloc(n_qubits)
    progY = Program()
    qubitsY = progY.qalloc(n_qubits)
    
    if spin == "+" :
        progX.apply(X, qubitsX[2*n_impurity])
        progY.apply(RY(np.pi), qubitsY[2*n_impurity]) #-iY
        for wire in range(2 * n_impurity) : 
            progX.apply(Z, qubitsX[wire])
            progY.apply(Z, qubitsY[wire])
    else :
        progX.apply(X, qubitsX[2*n_impurity + 1])
        progY.apply(RY(np.pi), qubitsY[2*n_impurity + 1])
        for wire in range(2 * n_impurity) :
            progX.apply(Z, qubitsX[wire])
            progY.apply(Z, qubitsY[wire])

    circuitX = progX.to_circ()
    circuitY = progY.to_circ()
    return circuitX, circuitY

XYarray (2 element array) below refers to what we are calculating, either the pauli string that features an X or the one that features a Y for $c_i$, and the pauli string that features an X or the one that features a Y for $c_j^{\dagger}$ for the computation of the unitaries whose sum gives us the green function. For example if XYarray = ["X", "Y"], we calculate $e^{i H t} (X_{2i} \prod_{j<2i} Z_j) e^{- i H t}(-iY_{2i} \prod_{j<2i} Z_j)$ (if both the spins are up)

In [12]:
def circuitGreen(n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray) :
    """Calcule le circuit quantique associé à la fonction de Green."""
    prog = Program()
    qubits = prog.qalloc(n_qubits)
    circuit = prog.to_circ()

    circuitX_c, circuitY_c = cMapping(n_qubits, i, sigma_i)
    circuitX_cdag, circuitY_cdag = cDagMapping(n_qubits, j, sigma_j)
    for i in range(N) : #Trotterization to compute exp(iHt)
        for term in h.terms :
            circuit = circuit + expPauli(n_qubits, term, t/N)

    if XYarray[0] == "X" :
        circuit = circuit + circuitX_c
    else :
        circuit = circuit + circuitY_c

    for i in range(N) : #2nd Trotterization to compute exp(-iHt)
        for term in h.terms :
            circuit = circuit + expPauli(n_qubits, term, -t/N)
    
    if XYarray[1] == "X" : #optimisation future circuitX_cdag = circuitX_c
        circuit = circuit + circuitX_cdag
    else :
        circuit = circuit + circuitY_cdag

    return circuit

circuitGreen(4, 0, "+", 0, "-", AndersonHamiltonian, 3, 6, ["X","Y"]).display()


1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0
1
0
0
2
2
2
2
0
0


It is left then to calculate the overlap between $\exp{(iHt)} c_{j,\sigma} \exp{(-iHt)} c_{i, \sigma'}^{\dagger} \ket{\psi_{0}}$ and $\ket{\psi_{0}}$ with a Hadamard test. For that we need to build a custom gate : control U that controls the ancillary qubit. <h3> I don't know how to do that since U is coded as a circuit and not as a gate or QRoutine so we code the hadamard test as if we had it available </h3>

In [13]:
def controlU(n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray) :
    """Applique le circuit U de la fonction de Green si l'état d'entrée est |1>, ne fait rien si c'est |0>."""
    prog = Program()
    qubits = prog.qalloc(n_qubits + 1) #+1 for the ancillary control qubit
    U = circuitGreen(n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray)
    return U.ctrl()(qubits[0], qubits[1:n_qubits])

In [14]:

def realHTest(psi0, n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray) :
    """Calcule la partie réelle de la valeur moyenne de U associée à un état initial. """
    prog = Program()
    qubits = prog.qalloc(n_qubits + 1) #we add an ancilla qubit
    prog.apply(H, qubits[0])
    #il reste à initialiser l'état fondamental sur lequel agit U
    prog = prog.apply(controlU(n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray), qubits[0], qubits[1:n_qubits])
    prog.apply(H, qubits[0])

    circuit = prog.to_circ()
    job = circuit.to_job(observable = Observable(1, pauli_terms=[Term(1, 'Z', [0])]))

def imHTest(psi0, n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray) :
    """"Calcule la partie imaginaire de la valeur moyenne de U associée à un état initial. """
    prog = Program()
    qubits = prog.qalloc(n_qubits + 1) #we add an ancilla qubit
    prog.apply(H, qubits[0])
    prog.apply(S.dag, qubits[0])
    #il reste à initialiser l'état fondamental sur lequel agit U
    prog = prog.apply(controlU(n_qubits, i, sigma_i, j, sigma_j, h, t, N, XYarray), qubits[0], qubits[1:n_qubits])
    prog.apply(H, qubits[0])

    circuit = prog.to_circ()
    job = circuit.to_job(observable = Observable(1, pauli_terms=[Term(1, 'Z', [0])]))




<h2> Ce qu'il reste à faire / à régler : </h2> 

- implémenter la porte custom Controlled U (pour les tests de Hadamard) où U correspond à la porte associée au circuit circuitGreen -- Pb : l'output de circuitGreen qui donne U est de classe Circuit, donc pas évident de créer une classe avec

- execution du job (qpu.submit()) qui crash le kernel à chaque fois et ne marche même pas avec le code fourni dans le documentation officielle de myQLM [sur mon ordinateur en tout cas]

- Comment initaliser un état dans un circuit, comme l'état fondamental calculé par le HVA : à implémenter dans les tests Hadamard

